- Make sure to set up your Street View Static API credentials. For more information, visit: https://developers.google.com/maps/documentation/streetview/usage-and-billing
- Please be aware that Street View Static API would charge your Google Account once you exceed free queries limit.

In [1]:
import base64
import hashlib
import hmac
import json
import logging
import urllib
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any
from urllib.parse import urlparse
import time

import geopandas as gpd
import pandas as pd
from tqdm import tqdm

In [2]:
class GSVDownloader:
    """A class to handle Google Street View image downloads with retry logic and progress tracking."""
    
    def __init__(self, api_key: str, secret: str, base_save_dir: Path = Path("./output/1")):
        """
        Initialize the GSV downloader.
        
        Args:
            api_key: Google Street View API key.
            secret: URL signing secret.
            base_save_dir: Base directory for saving images.
        """
        self.api_key = api_key
        self.secret = secret
        self.base_save_dir = base_save_dir
        self.default_params = {
            'size': [512, 512],
            'direction': None,
            'zoom': 90,
            'vertical_angle': 10,
            'rad': 200,
            'source': "outdoor",
            'max_retries': 3,
            'retry_delay': 2
        }
        self.logger = logging.getLogger(__name__)
    
    def _create_url(self, location: List[float], params: Dict[str, Any]) -> str:
        """
        Construct URL for Google Street View API.
        For more information: https://developers.google.com/maps/documentation/streetview/request-streetview
        
        Args:
            location: [latitude, longitude] coordinates.
            params: Additional parameters for the request.
            
        Returns:
            Complete signed URL
        """
        # Validate location format
        if not isinstance(location, (list, tuple)) or len(location) != 2:
            raise ValueError("location must be a list or tuple with two elements [lat, lng]")
        
        # URL endpoint
        url_endpoint = 'https://maps.googleapis.com/maps/api/streetview?'
        
        # Build parameters
        request_params = {
            'location': f"{location[1]},{location[0]}",
            'size': f"{params['size'][0]}x{params['size'][1]}",
            'heading': params.get('direction'),
            'fov': params.get('zoom', 90),
            'pitch': params.get('vertical_angle', 0),
            'radius': params.get('rad', 50),
            'source': params.get('source', 'outdoor'),
            'key': self.api_key
        }
        
        # Remove None values
        request_params = {k: v for k, v in request_params.items() if v is not None}
        
        # Create query string
        query_string = '&'.join([f"{key}={value}" for key, value in request_params.items()])
        
        return url_endpoint + query_string
    
    def _sign_url(self, input_url: str) -> str:
        """
        Sign a request URL with a URL signing secret.
        
        Args:
            input_url: The URL to sign.
            
        Returns:
            The signed request URL.
        """
        # Parse the URL
        url = urlparse(input_url)
        
        # Only need to sign the path + query part of the string
        url_to_sign = f"{url.path}?{url.query}"
        
        # Decode the secret key into its binary format
        decoded_key = base64.urlsafe_b64decode(self.secret)
        
        # Create a signature using the secret key and the URL-encoded string
        signature = hmac.new(decoded_key, url_to_sign.encode(), hashlib.sha1)
        
        # Encode the binary signature into base64 for use within a URL
        encoded_signature = base64.urlsafe_b64encode(signature.digest()).decode()
        
        # Construct the signed URL
        return f"{input_url}&signature={encoded_signature}"
    
    def _download_image(self, url: str, file_path: Path, max_retries: int = 3, retry_delay: int = 2) -> bool:
        """
        Download an image with retry logic.
        
        Args:
            url: Image URL to download.
            file_path: Path to save the image.
            max_retries: Maximum number of retry attempts.
            retry_delay: Delay between retries in seconds.
            
        Returns:
            True if download was successful, False otherwise.
        """
        for attempt in range(max_retries):
            try:
                urllib.request.urlretrieve(url, file_path)
                return True
            except Exception as e:
                if attempt < max_retries - 1:
                    self.logger.warning(f"Attempt {attempt + 1} failed for {file_path.name}: {e}. Retrying...")
                    time.sleep(retry_delay)
                    continue
                else:
                    self.logger.error(f"Failed to download image {file_path.name} after {max_retries} attempts: {e}")
                    return False
        return False
    
    def download_images(self, 
                       df: pd.DataFrame, 
                       coordinates_col: str = 'coordinates',
                       id_col: str = 'id',
                       folder_name: str = "raw",
                       size: List[int] = None,
                       direction: Optional[int] = None,
                       zoom: Optional[int] = None,
                       vertical_angle: Optional[int] = None,
                       rad: Optional[int] = None,
                       source: str = None,
                       max_retries: Optional[int] = None,
                       retry_delay: Optional[int] = None) -> pd.DataFrame:
        """
        Download Google Street View images for multiple locations.
        
        Args:
            df: DataFrame containing location data.
            coordinates_col: Column name containing [lat, lng] coordinates.
            id_col: Column name containing unique identifiers.
            folder_name: Subfolder name for organizing images.
            size: Image size as [width, height] in pixels.
            direction: Compass heading of the camera (0-360).
            zoom: Horizontal field of view in degrees (max 120).
            vertical_angle: Up/down angle of camera (-90 to 90).
            rad: Search radius in meters (centered on location).
            source: Limits searches to specific sources ("default" or "outdoor").
            max_retries: Maximum number of retry attempts.
            retry_delay: Delay between retries in seconds.
            
        Returns:
            DataFrame containing metadata for failed downloads.
        """
        # Merge default and provided parameters
        params = {
            'size': size if size is not None else self.default_params['size'],
            'direction': direction,
            'zoom': zoom if zoom is not None else self.default_params['zoom'],
            'vertical_angle': vertical_angle if vertical_angle is not None else self.default_params['vertical_angle'],
            'rad': rad if rad is not None else self.default_params['rad'],
            'source': source if source is not None else self.default_params['source'],
            'max_retries': max_retries if max_retries is not None else self.default_params['max_retries'],
            'retry_delay': retry_delay if retry_delay is not None else self.default_params['retry_delay']
        }
        
        self.logger.info(f"Starting GSV download with parameters: {params}")
        
        # Confirm download operation
        total_images = len(df)
        if total_images == 0:
            self.logger.warning("No images to download.")
            return pd.DataFrame()
        
        self.logger.info(f"Preparing to download {total_images} images.")
        proceed = input("Do you want to proceed? (y/n): ")
        if proceed.lower() != 'y':
            self.logger.info("Download cancelled by user.")
            return pd.DataFrame()
        
        # Create save directory
        save_dir = self.base_save_dir / folder_name

        # Create directory if it doesn't exist
        save_dir.mkdir(parents=True, exist_ok=True)
        self.logger.info(f"Save directory: {save_dir}")
        
        # Check for existing images
        existing_files = {int(f.stem.split('_')[0]) for f in save_dir.glob("*_GSV.jpeg")}
        images_to_download = df[~df[id_col].isin(existing_files)]
        
        if len(images_to_download) < total_images:
            self.logger.info(f"Found {len(existing_files)} existing images. Downloading {len(images_to_download)} new images.")
        
        failed_downloads = []
        
        # Download images with progress bar
        with tqdm(total=len(images_to_download), desc="Downloading GSV images") as pbar:
            for _, row in images_to_download.iterrows():
                try:
                    # Create signed URL
                    url = self._create_url(row[coordinates_col], params)
                    signed_url = self._sign_url(url)
                    
                    # Create file path
                    file_path = save_dir / f"{row[id_col]}_GSV.jpeg"
                    
                    # Download image
                    success = self._download_image(
                        signed_url, 
                        file_path, 
                        max_retries=params['max_retries'],
                        retry_delay=params['retry_delay']
                    )
                    
                    if success:
                        self.logger.debug(f"Successfully downloaded image for ID {row[id_col]}")
                    else:
                        self.logger.warning(f"Failed to download image for ID {row[id_col]}")
                        failed_downloads.append(row.to_dict())
                    
                except Exception as e:
                    self.logger.error(f"Error processing image {row[id_col]}: {e}")
                    failed_downloads.append(row.to_dict())
                
                pbar.update(1)
                time.sleep(0.1)  # Small delay to respect rate limits
        
        # Report results
        successful = len(images_to_download) - len(failed_downloads)
        self.logger.info(f"Download completed. Successful: {successful}, Failed: {len(failed_downloads)}")
        
        if failed_downloads:
            self.logger.warning(f"{len(failed_downloads)} images failed to download")
            return pd.DataFrame(failed_downloads)
        
        self.logger.info("All images downloaded successfully.")
        return pd.DataFrame()

def prepare_building_data(penjaringan_path: str, buildings_path: str) -> gpd.GeoDataFrame:
    """
    Prepare building data for GSV download by clipping to area and calculating centroids.
    
    Args:
        penjaringan_path: Path to administrative area file.
        buildings_path: Path to buildings file.
        
    Returns:
        GeoDataFrame with prepared building data.
    """
    # Load administrative area
    penjaringan_gdf = gpd.read_parquet(penjaringan_path)[["geometry"]]
    
    # Load buildings and clip to area
    buildings_gdf = gpd.read_parquet(buildings_path)
    buildings_clipped = buildings_gdf.clip(penjaringan_gdf.dissolve())
    
    # Prepare coordinates for GSV API
    buildings_clipped["id"] = buildings_clipped.index + 1
    buildings_clipped["centroid"] = buildings_clipped.centroid
    
    # Convert to geographic CRS for coordinates
    centroids_4326 = buildings_clipped["centroid"].to_crs("EPSG:4326")
    buildings_clipped["lon"] = centroids_4326.x
    buildings_clipped["lat"] = centroids_4326.y
    buildings_clipped["coordinates"] = buildings_clipped.apply(
        lambda row: [row['lat'], row['lon']], axis=1
    )
    
    # Clean up temporary columns
    buildings_clipped = buildings_clipped.drop(["centroid", "lon", "lat"], axis=1)
    
    return buildings_clipped


def load_gsv_credentials(credentials_path: str) -> Dict[str, str]:
    """
    Load Google Street View API credentials from JSON file.
    
    Args:
        credentials_path: Path to credentials JSON file.
        
    Returns:
        Dictionary with API key and secret.
    """
    with open(credentials_path) as f:
        return json.load(f)

You might want to query a few buildings first in your study area to find the most suitable parameters (i.e., size, direction, zoom, vertical_angle, rad).

In [3]:
# Configuration
PENJARINGAN_PATH = "./input/adm_penjaringan.gzip"
BUILDINGS_PATH = "./output/0/buildings.gzip"
CREDENTIALS_PATH = "./helper/gsv_token.json"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("./temp/1.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [4]:
# Prepare building data
logger.info("Preparing building data...")
buildings_data = prepare_building_data(PENJARINGAN_PATH, BUILDINGS_PATH)

# Load credentials
logger.info("Loading API credentials...")
credentials = load_gsv_credentials(CREDENTIALS_PATH)

# Initialize downloader
downloader = GSVDownloader(
    api_key=credentials['api_key'],
    secret=credentials['secret']
)

2025-10-27 15:16:12,171 - INFO - Preparing building data...
2025-10-27 15:16:27,515 - INFO - Loading API credentials...


In [ ]:
# Experiment with first 10 buildings (adjust parameters as needed)
folder_name = "experimental"
logger.info("Starting image download...")

failed_downloads = downloader.download_images(
    df=buildings_data.head(10),  # Use buildings_data for full dataset
    coordinates_col='coordinates',
    id_col='id',
    folder_name=folder_name,
    size=[512, 512],
    direction=None,
    zoom=120,
    vertical_angle=10,
    rad=200,
    source="outdoor",
    max_retries=3,
    retry_delay=2
)

# Retry failed downloads if any
if not failed_downloads.empty:
    logger.info(f"Retrying {len(failed_downloads)} failed downloads...")
    retry_failed = downloader.download_images(
        df=failed_downloads,
        coordinates_col='coordinates',
        id_col='id',
        folder_name=folder_name,
        size=[512, 512],
        direction=None,
        zoom=120,  # To experiment with
        vertical_angle=10,  # To experiment with
        rad=200,
        source="outdoor"
    )
    
    if not retry_failed.empty:
        logger.warning(f"Still have {len(retry_failed)} failed downloads after retry")
        return retry_failed

logger.info("GSV download pipeline completed successfully")

In [ ]:
# Full dataset download
folder_name = "raw"
logger.info("Starting image download...")
failed_downloads = downloader.download_images(
    df=buildings_data,
    coordinates_col='coordinates',
    id_col='id',
    folder_name=folder_name,
    size=[512, 512],
    direction=None,
    zoom=120,
    vertical_angle=10,
    rad=200,
    source="outdoor",
    max_retries=3,
    retry_delay=2
)

# Retry failed downloads if any
if not failed_downloads.empty:
    logger.info(f"Retrying {len(failed_downloads)} failed downloads...")
    retry_failed = downloader.download_images(
        df=failed_downloads,
        coordinates_col='coordinates',
        id_col='id',
        folder_name=folder_name,
        size=[512, 512],
        direction=None,
        zoom=120,
        vertical_angle=10,
        rad=200,
        source="outdoor"
    )
    
    if not retry_failed.empty:
        logger.warning(f"Still have {len(retry_failed)} failed downloads after retry")
        return retry_failed

logger.info("GSV download pipeline completed successfully")